# 三元组存储入门

本 Notebook 介绍 **Semantica 三元组存储(Triplet Store)**模块,它允许你使用业界标准的 RDF 三元组存储来存储和查询知识图谱数据。

## 什么是三元组存储?

三元组存储(或称 RDF 存储)是一种为存储和检索三元组而优化的数据库:**主语 - 谓语 - 宾语**。
例如:`Alice`(主语)`knows`(谓语)`Bob`(宾语)。

Semantica 支持:
- **Blazegraph**(默认,高性能)
- **Apache Jena**(非常适合推理)
- **RDF4J**(标准的 Java 框架)

In [ ]:
import sys
import os

# Add the project root to the path so we can import the local version of semantica
sys.path.append(os.path.abspath('../../'))

# If running in Google Colab, uncomment the following line to install dependencies
# !pip install -q semantica

In [ ]:
# Import the TripletStore class
from semantica.triplet_store import TripletStore
from semantica.semantic_extract.triplet_extractor import Triplet

## 1. 连接到存储

要使用三元组存储,你需要一个正在运行的后端。在本示例中,我们假设本地运行着一个 Blazegraph 实例。

> **注意:**如果你没有运行中的存储,下面的代码展示的是你*将会*如何连接。

In [ ]:
# Connect to a Blazegraph instance
# You can also use backend="jena" or backend="rdf4j"
store = TripletStore(
    backend="blazegraph",
    endpoint="http://localhost:9999/blazegraph"
)

# Check connection status
if hasattr(store._store_backend, 'connected') and store._store_backend.connected:
    print(f"Successfully connected to {store.backend_type} store at {store.endpoint}")
else:
    print(f"Warning: Could not connect to {store.backend_type} at {store.endpoint}")
    print("Operations requiring a live store connection will be skipped or fail.")

## 2. 创建三元组

我们使用 `Triplet` 类来定义数据。

In [ ]:
# Define a single triplet
triplet1 = Triplet(
    subject="http://example.org/Alice",
    predicate="http://xmlns.com/foaf/0.1/knows",
    object="http://example.org/Bob"
)

print(f"Created triplet: {triplet1.subject} -> {triplet1.predicate} -> {triplet1.object}")

## 3. 添加数据

你可以逐个添加三元组,也可以批量添加。

In [ ]:
from semantica.utils.exceptions import ProcessingError

try:
    # Add a single triplet
    store.add_triplet(triplet1)
    print("Added single triplet successfully.")

    # Create more triplets
    triplets = [
        Triplet(
            subject="http://example.org/Bob",
            predicate="http://xmlns.com/foaf/0.1/knows",
            object="http://example.org/Charlie"
        ),
        Triplet(
            subject="http://example.org/Charlie",
            predicate="http://xmlns.com/foaf/0.1/knows",
            object="http://example.org/David"
        )
    ]

    # Bulk add
    store.add_triplets(triplets)
    print("Added bulk triplets successfully.")

except ProcessingError as e:
    print(f"Operation skipped: {e}")
except Exception as e:
    print(f"An error occurred: {e}")

## 4. 查询数据(SPARQL)

使用 SPARQL 查询从存储中检索数据。

In [ ]:
# Simple query to get all triplets (limited to 10)
query = """
SELECT ?s ?p ?o
WHERE {
  ?s ?p ?o
}
LIMIT 10
"""

try:
    results = store.execute_query(query)
    print("Query Results:", results)
except ProcessingError as e:
    print(f"Query skipped: {e}")
except Exception as e:
    print(f"An error occurred: {e}")

## 5. 删除数据

当三元组不再需要时,将其移除。

In [ ]:
store.delete_triplet(triplet1)
print("Deleted triplet1")

## 下一步

请查看 `cookbook/advanced` 文件夹中的**高级三元组存储**指南,了解更复杂的操作,例如批量加载优化、事务和高级 SPARQL 特性。